Libraries

In [2]:
import yfinance as yf
import numpy as np
import statsmodels.api as sm

from statsmodels.tsa.stattools import coint
from itertools import combinations
from tqdm import tqdm
import pandas as pd
import requests
from io import StringIO

Data Gathering

In [3]:
def download_sp500_tickers():
    """ Obtain the tickers of the sp500 stocks from the Wikipedia table"""
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    response = requests.get(url, headers=headers)
    response.raise_for_status()

    tables = pd.read_html(StringIO(response.text))
    table = tables[0]

    tickers = table["Symbol"].tolist()
    tickers = [t.replace(".", "-") for t in tickers]

    df = pd.DataFrame({"ticker": tickers})

    df.to_csv("sp500_tickers.csv", index=False)

    return

def download_prices(
    tickers,
    start="2018-01-01",
    end="2025-01-01",
    output_path="prices.csv",
    filtered_output_path="prices_filtered.csv",
    min_obs=500,
    max_missing_ratio=0.10
):
    data = yf.download(
        tickers,
        start=start,
        end=end,
        auto_adjust=True,
        progress=False
    )

    prices = data["Close"]

    if isinstance(prices, pd.Series):
        prices = prices.to_frame()

    prices = prices.dropna(how="all")
    prices = prices.ffill()

    # Save raw cleaned prices
    prices.to_csv(output_path)
    print(f"Raw prices saved in {output_path}")
    print(f"Raw shape: {prices.shape}")

    def filter_assets(prices, min_obs=500, max_missing_ratio=0.10):
        valid_tickers = []

        for ticker in prices.columns:
            s = prices[ticker]

            missing_ratio = s.isna().mean()
            n_obs = s.dropna().shape[0]

            if n_obs >= min_obs and missing_ratio <= max_missing_ratio:
                valid_tickers.append(ticker)

        return prices[valid_tickers]


    # Filter assets immediately
    filtered_prices = filter_assets(
        prices,
        min_obs=min_obs,
        max_missing_ratio=max_missing_ratio
    )

    # Save filtered prices
    filtered_prices.to_csv(filtered_output_path)
    print(f"Filtered prices saved in {filtered_output_path}")
    print(f"Filtered shape: {filtered_prices.shape}")

    return filtered_prices

if __name__ == "__main__":

    download_sp500_tickers()

    tickers = pd.read_csv("sp500_tickers.csv")["ticker"].tolist()

    print(f"Number of tickers: {len(tickers)}")
    print(tickers[:10])

    prices = download_prices(
        tickers=tickers,
        start="2018-01-01",
        end="2025-01-01",
        output_path="prices.csv",
        filtered_output_path="prices_filtered.csv",
        min_obs=500,
        max_missing_ratio=0.10
    )

    print("Prices downloaded and filtered")

Number of tickers: 503
['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A']


$Q: possibly delisted; no price data found  (1d 2018-01-01 -> 2025-01-01) (Yahoo error = "Data doesn't exist for startDate = 1514782800, endDate = 1735707600")
$SNDK: possibly delisted; no price data found  (1d 2018-01-01 -> 2025-01-01) (Yahoo error = "Data doesn't exist for startDate = 1514782800, endDate = 1735707600")

2 Failed downloads:
['Q', 'SNDK']: possibly delisted; no price data found  (1d 2018-01-01 -> 2025-01-01) (Yahoo error = "Data doesn't exist for startDate = 1514782800, endDate = 1735707600")


Raw prices saved in prices.csv
Raw shape: (1761, 503)
Filtered prices saved in prices_filtered.csv
Filtered shape: (1761, 478)
Prices downloaded and filtered


In [ ]:
def load_prices(input_path="prices.csv"):
    prices = pd.read_csv(input_path, index_col=0, parse_dates=True)
    return prices

def estimate_hedge_ratio(y, x):
    X = sm.add_constant(x)
    model = sm.OLS(y, X).fit()

    alpha = model.params.iloc[0]
    beta = model.params.iloc[1]

    return alpha, beta


def calculate_half_life(spread):
    "Intenta medir cuanto tarda en volver hacia su media cuando se desvía"
    spread_lag = spread.shift(1)
    spread_delta = spread - spread_lag

    df = pd.concat([spread_delta, spread_lag], axis=1).dropna()
    df.columns = ["delta", "lag"]

    if len(df) < 50:
        return np.nan

    X = sm.add_constant(df["lag"])
    model = sm.OLS(df["delta"], X).fit()

    lambda_ = model.params["lag"]

    if lambda_ >= 0:
        return np.nan

    half_life = -np.log(2) / lambda_

    return half_life


def test_pair(prices, ticker_a, ticker_b, min_pair_obs=500):
    pair_prices = prices[[ticker_a, ticker_b]].dropna()

    if len(pair_prices) < min_pair_obs:
        return None

    y = pair_prices[ticker_a]
    x = pair_prices[ticker_b]

    try:
        #Test de Cointegración Engler-Grangler 
        # H0: No hay cointegración 
        # H1: Sí hay cointegración
        score, pvalue, critical_values = coint(y, x) 

        alpha, beta = estimate_hedge_ratio(y, x)
        spread = y - alpha - beta * x

        half_life = calculate_half_life(spread)

        spread_mean = spread.mean()
        spread_std = spread.std()

        if spread_std == 0:
            return None

        zscore_last = (spread.iloc[-1] - spread_mean) / spread_std

        return {
            "ticker_a": ticker_a,
            "ticker_b": ticker_b,
            "pvalue": pvalue,
            "test_stat": score,
            "crit_1pct": critical_values[0],
            "crit_5pct": critical_values[1],
            "crit_10pct": critical_values[2],
            "alpha": alpha,
            "beta": beta,
            "half_life": half_life,
            "zscore_last": zscore_last,
            "n_obs": len(pair_prices)
        }

    except Exception as e:
        return None


def run_cointegration_screener(
    prices_path="prices_filtered.csv",
    min_pair_obs=500,
    output_path="cointegration_results.csv"
):
    
    print(f"Loading prices from {prices_path}...")
    prices = load_prices(prices_path)

    valid_tickers = list(prices.columns)

    print(f"Valid tickers: {len(valid_tickers)}")

    pairs = list(combinations(valid_tickers, 2))

    print(f"Testing {len(pairs)} pairs...")

    results = []

    for ticker_a, ticker_b in tqdm(pairs):
        result = test_pair(
            prices,
            ticker_a,
            ticker_b,
            min_pair_obs=min_pair_obs
        )

        if result is not None:
            results.append(result)

    results_df = pd.DataFrame(results)

    if results_df.empty:
        print("No valid results.")
        return results_df

    results_df = results_df.sort_values("pvalue")

    results_df.to_csv(output_path, index=False)

    print(f"Saved results to {output_path}")

    return results_df

In [ ]:
if __name__ == "__main__":

    results = run_cointegration_screener(
        prices_path="prices.csv",
        min_obs=500,
        max_missing_ratio=0.10,
        min_pair_obs=500,
        output_path="cointegration_results.csv"
    )

    print(results.head(20))

$SNDK: possibly delisted; no price data found  (1d 2018-01-01 -> 2025-01-01) (Yahoo error = "Data doesn't exist for startDate = 1514782800, endDate = 1735707600")
$Q: possibly delisted; no price data found  (1d 2018-01-01 -> 2025-01-01) (Yahoo error = "Data doesn't exist for startDate = 1514782800, endDate = 1735707600")

2 Failed downloads:
['SNDK', 'Q']: possibly delisted; no price data found  (1d 2018-01-01 -> 2025-01-01) (Yahoo error = "Data doesn't exist for startDate = 1514782800, endDate = 1735707600")


Filtering assets...
Valid tickers: 478
Testing 114003 pairs...


  0%|          | 370/114003 [00:29<2:30:47, 12.56it/s]


KeyboardInterrupt: 